Assignment-1

In [ ]:
# sanity_check.py

import sys
import platform
import os

print("=== HPC Environment Sanity Check ===\n")

# Python version
print("Python Version:")
print(sys.version, "\n")

# Platform details
print("System Information:")
print(f"System: {platform.system()}")
print(f"Node Name: {platform.node()}")
print(f"Release: {platform.release()}")
print(f"Processor: {platform.processor()}\n")

# CPU count
print("CPU Information:")
print(f"Number of CPUs: {os.cpu_count()}\n")

# Simple test computation
print("Running a simple test...")
result = sum(range(1, 1001))
print(f"Sum of numbers from 1 to 1000 = {result}\n")

print("✅ Environment check completed successfully!")

=== HPC Environment Sanity Check ===

Python Version:
3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0] 

System Information:
System: Linux
Node Name: 476d4e11d4a0
Release: 6.6.113+
Processor: x86_64

CPU Information:
Number of CPUs: 2

Running a simple test...
Sum of numbers from 1 to 1000 = 500500

✅ Environment check completed successfully!


Assignment-2

In [ ]:
# vector_operations.py

import time
import random
import cProfile
import pstats
import io
import tracemalloc

# Generate random vectors
def generate_vectors(n):
    a = [random.random() for _ in range(n)]
    b = [random.random() for _ in range(n)]
    return a, b

# Element-wise addition (pure Python loop)
def vector_add(a, b):
    result = []
    for i in range(len(a)):
        result.append(a[i] + b[i])
    return result

# Dot product (pure Python loop)
def dot_product(a, b):
    result = 0.0
    for i in range(len(a)):
        result += a[i] * b[i]
    return result

def main():
    n = 10**6  # vector size (adjust if needed)

    print("Generating vectors...")
    a, b = generate_vectors(n)

    # Start memory tracking
    tracemalloc.start()

    # Time: Vector Addition
    start = time.time()
    c = vector_add(a, b)
    end = time.time()
    print(f"Vector Addition Time: {end - start:.4f} seconds")

    # Time: Dot Product
    start = time.time()
    dp = dot_product(a, b)
    end = time.time()
    print(f"Dot Product Time: {end - start:.4f} seconds")

    print(f"Dot Product Result: {dp}")

    # Memory usage
    current, peak = tracemalloc.get_traced_memory()
    print(f"Current Memory Usage: {current / 10**6:.2f} MB")
    print(f"Peak Memory Usage: {peak / 10**6:.2f} MB")
    tracemalloc.stop()

# Profiling wrapper
def profile_code():
    pr = cProfile.Profile()
    pr.enable()

    main()

    pr.disable()
    s = io.StringIO()
    ps = pstats.Stats(pr, stream=s).sort_stats('cumulative')
    ps.print_stats(10)  # Top 10 functions
    print("\n=== Profiling Results ===")
    print(s.getvalue())

if __name__ == "__main__":
    profile_code()

Generating vectors...
Vector Addition Time: 4.8649 seconds
Dot Product Time: 1.1389 seconds
Dot Product Result: 250286.96241377798
Current Memory Usage: 32.45 MB
Peak Memory Usage: 32.45 MB

=== Profiling Results ===
         3001265 function calls (3001255 primitive calls) in 6.712 seconds

   Ordered by: cumulative time
   List reduced from 137 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        7    0.000    0.000   12.195    1.742 /usr/lib/python3.12/asyncio/base_events.py:1922(_run_once)
        7    0.047    0.007    6.685    0.955 /usr/lib/python3.12/selectors.py:451(select)
        6    3.920    0.653    5.188    0.865 {built-in method time.sleep}
        6    0.337    0.056    1.382    0.230 {method 'poll' of 'select.epoll' objects}
  1000011    1.331    0.000    1.331    0.000 {method 'append' of 'list' objects}
        1    0.243    0.243    0.391    0.391 /tmp/ipykernel_5777/2169932067.py:11(generate_vectors)
      

Assignment-3

In [ ]:
# numba_vector_parallel.py

import numpy as np
import time
from numba import njit, prange

# -----------------------------
# Serial Version
# -----------------------------
@njit
def serial_compute(A, B, alpha):
    n = len(A)
    C = np.empty(n, dtype=np.float64)
    for i in range(n):
        C[i] = alpha * A[i] + B[i]
    return C

# -----------------------------
# Parallel Version using prange
# -----------------------------
@njit(parallel=True)
def parallel_compute(A, B, alpha):
    n = len(A)
    C = np.empty(n, dtype=np.float64)
    for i in prange(n):   # parallel loop
        C[i] = alpha * A[i] + B[i]
    return C

# -----------------------------
# Benchmark Function
# -----------------------------
def benchmark():
    sizes = [10**5, 5*10**5, 10**6, 5*10**6]
    alpha = 2.5

    print("=== Parallel Vector Computation (Numba) ===\n")

    for n in sizes:
        print(f"Vector Size: {n}")

        A = np.random.rand(n)
        B = np.random.rand(n)

        # Warm-up (JIT compilation happens here)
        serial_compute(A, B, alpha)
        parallel_compute(A, B, alpha)

        # Serial timing
        start = time.time()
        C1 = serial_compute(A, B, alpha)
        serial_time = time.time() - start

        # Parallel timing
        start = time.time()
        C2 = parallel_compute(A, B, alpha)
        parallel_time = time.time() - start

        # Speedup
        speedup = serial_time / parallel_time if parallel_time > 0 else 0

        print(f"Serial Time   : {serial_time:.4f} sec")
        print(f"Parallel Time : {parallel_time:.4f} sec")
        print(f"Speedup       : {speedup:.2f}x\n")

# -----------------------------
# Main
# -----------------------------
if __name__ == "__main__":
    benchmark()

=== Parallel Vector Computation (Numba) ===

Vector Size: 100000
Serial Time   : 0.0002 sec
Parallel Time : 0.0002 sec
Speedup       : 1.27x

Vector Size: 500000
Serial Time   : 0.0010 sec
Parallel Time : 0.0009 sec
Speedup       : 1.09x

Vector Size: 1000000
Serial Time   : 0.0020 sec
Parallel Time : 0.0018 sec
Speedup       : 1.10x

Vector Size: 5000000
Serial Time   : 0.0229 sec
Parallel Time : 0.0257 sec
Speedup       : 0.89x



Assignment-4

In [ ]:
# parallel_vector_addition.py

import numpy as np
import time
from numba import njit, prange

# -----------------------------
# Serial Vector Addition
# -----------------------------
@njit
def serial_add(A, B):
    n = len(A)
    C = np.empty(n, dtype=np.float64)
    for i in range(n):
        C[i] = A[i] + B[i]
    return C

# -----------------------------
# Parallel Vector Addition
# -----------------------------
@njit(parallel=True)
def parallel_add(A, B):
    n = len(A)
    C = np.empty(n, dtype=np.float64)
    for i in prange(n):   # parallel loop (OpenMP style)
        C[i] = A[i] + B[i]
    return C

# -----------------------------
# Benchmark Function
# -----------------------------
def benchmark():
    sizes = [10**5, 5*10**5, 10**6, 5*10**6]

    print("=== Parallel Vector Addition ===\n")

    for n in sizes:
        print(f"Vector Size: {n}")

        # Initialize vectors
        A = np.random.rand(n)
        B = np.random.rand(n)

        # Warm-up (important for JIT)
        serial_add(A, B)
        parallel_add(A, B)

        # Serial execution time
        start = time.time()
        C1 = serial_add(A, B)
        serial_time = time.time() - start

        # Parallel execution time
        start = time.time()
        C2 = parallel_add(A, B)
        parallel_time = time.time() - start

        # Speedup calculation
        speedup = serial_time / parallel_time if parallel_time > 0 else 0

        print(f"Serial Time   : {serial_time:.4f} sec")
        print(f"Parallel Time : {parallel_time:.4f} sec")
        print(f"Speedup       : {speedup:.2f}x\n")

# -----------------------------
# Main
# -----------------------------
if __name__ == "__main__":
    benchmark()

=== Parallel Vector Addition ===

Vector Size: 100000
Serial Time   : 0.0002 sec
Parallel Time : 0.0002 sec
Speedup       : 1.01x

Vector Size: 500000
Serial Time   : 0.0010 sec
Parallel Time : 0.0010 sec
Speedup       : 0.96x

Vector Size: 1000000
Serial Time   : 0.0021 sec
Parallel Time : 0.0020 sec
Speedup       : 1.04x

Vector Size: 5000000
Serial Time   : 0.0326 sec
Parallel Time : 0.0403 sec
Speedup       : 0.81x



Assignment-5

In [ ]:
%%writefile fib_recursive.c
#include <stdio.h>
#include <time.h>

// Recursive Fibonacci function
long long fib(int n) {
    if (n <= 1)
        return n;
    return fib(n - 1) + fib(n - 2);
}

int main() {
    int n = 40;  // change value here

    clock_t start, end;

    start = clock();
    long long result = fib(n);
    end = clock();

    double time_taken = (double)(end - start) / CLOCKS_PER_SEC;

    printf("Fibonacci(%d) = %lld\n", n, result);
    printf("Execution Time = %.6f seconds\n", time_taken);

    return 0;
}

Writing fib_recursive.c


In [ ]:
!gcc fib_recursive.c -o fib
!./fib

Fibonacci(40) = 102334155
Execution Time = 1.103422 seconds


Assignment-6

In [ ]:
!lscpu

Architecture:                x86_64
  CPU op-mode(s):            32-bit, 64-bit
  Address sizes:             46 bits physical, 48 bits virtual
  Byte Order:                Little Endian
CPU(s):                      2
  On-line CPU(s) list:       0,1
Vendor ID:                   GenuineIntel
  Model name:                Intel(R) Xeon(R) CPU @ 2.20GHz
    CPU family:              6
    Model:                   79
    Thread(s) per core:      2
    Core(s) per socket:      1
    Socket(s):               1
    Stepping:                0
    BogoMIPS:                4400.44
    Flags:                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pg
                             e mca cmov pat pse36 clflush mmx fxsr sse sse2 ss h
                             t syscall nx pdpe1gb rdtscp lm constant_tsc rep_goo
                             d nopl xtopology nonstop_tsc cpuid tsc_known_freq p
                             ni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2ap
                   

In [ ]:
!apt-get install numactl -y
!numactl --hardware

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  numactl
0 upgraded, 1 newly installed, 0 to remove and 7 not upgraded.
Need to get 36.8 kB of archives.
After this operation, 140 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 numactl amd64 2.0.14-3ubuntu2 [36.8 kB]
Fetched 36.8 kB in 0s (88.6 kB/s)
Selecting previously unselected package numactl.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../numactl_2.0.14-3ubuntu2_amd64.deb ...
Unpacking numactl (2.0.14-3ubuntu2) ...
Setting up numactl (2.0.14-3ubuntu2) ...
Processing triggers for man-db (2.10.2-1) ...
available: 1 nodes (0)
node 0 cpus: 0 1
node 0 size: 12975 MB
node 0 free: 8167 MB
node distances:
node   0 
  0:  10 


Assignment-7

In [ ]:

import numpy as np
import time

# -----------------------------
# Scalar (Normal Python Loop)
# -----------------------------
def scalar_add(A, B):
    C = []
    for i in range(len(A)):
        C.append(A[i] + B[i])
    return C

# -----------------------------
# SIMD-style (NumPy Vectorized)
# -----------------------------
def simd_add(A, B):
    return A + B   # vectorized → uses SIMD internally

# -----------------------------
# Benchmark
# -----------------------------
def benchmark():
    sizes = [10**5, 10**6, 5*10**6]

    print("=== SIMD Vector Addition (Python) ===\n")

    for n in sizes:
        print(f"Vector Size: {n}")

        # Initialize data
        A = np.random.rand(n)
        B = np.random.rand(n)

        # Convert to list for scalar version
        A_list = A.tolist()
        B_list = B.tolist()

        # -----------------------------
        # Scalar timing
        # -----------------------------
        start = time.time()
        C1 = scalar_add(A_list, B_list)
        scalar_time = time.time() - start

        # -----------------------------
        # SIMD (NumPy) timing
        # -----------------------------
        start = time.time()
        C2 = simd_add(A, B)
        simd_time = time.time() - start

        # Speedup
        speedup = scalar_time / simd_time if simd_time > 0 else 0

        print(f"Scalar Time : {scalar_time:.4f} sec")
        print(f"SIMD Time   : {simd_time:.4f} sec")
        print(f"Speedup     : {speedup:.2f}x\n")

# -----------------------------
# Main
# -----------------------------
if __name__ == "__main__":
    benchmark()

=== SIMD Vector Addition (Python) ===

Vector Size: 100000
Scalar Time : 0.0085 sec
SIMD Time   : 0.0003 sec
Speedup     : 30.46x

Vector Size: 1000000
Scalar Time : 0.0978 sec
SIMD Time   : 0.0045 sec
Speedup     : 21.74x

Vector Size: 5000000
Scalar Time : 0.5131 sec
SIMD Time   : 0.0240 sec
Speedup     : 21.35x



Assignment-8

In [ ]:
# amdahl_law_demo.py

import time
import multiprocessing as mp

# -----------------------------
# Serial Work (Bottleneck)
# -----------------------------
def serial_work(n):
    total = 0
    for i in range(n):
        total += i * i
    return total

# -----------------------------
# Parallel Work
# -----------------------------
def parallel_work(n):
    total = 0
    for i in range(n):
        total += i * i
    return total

# -----------------------------
# Worker Function
# -----------------------------
def worker(n):
    return parallel_work(n)

# -----------------------------
# Run Experiment
# -----------------------------
def run_experiment(n, num_processes):
    start = time.time()

    # Serial part (cannot be parallelized)
    serial_work(n // 4)

    # Parallel part
    pool = mp.Pool(processes=num_processes)
    tasks = [n // num_processes] * num_processes
    results = pool.map(worker, tasks)
    pool.close()
    pool.join()

    end = time.time()
    return end - start

# -----------------------------
# Main
# -----------------------------
if __name__ == "__main__":
    n = 10**7
    thread_counts = [1, 2, 4, 8]

    print("=== Amdahl's Law Experiment ===\n")

    times = []

    # Run for different thread counts
    for t in thread_counts:
        exec_time = run_experiment(n, t)
        times.append(exec_time)
        print(f"Threads: {t}, Time: {exec_time:.4f} sec")

    print("\n=== Performance Metrics ===\n")

    T1 = times[0]

    for i, t in enumerate(thread_counts):
        speedup = T1 / times[i]
        efficiency = speedup / t

        print(f"Threads: {t}")
        print(f"  Speedup   : {speedup:.2f}")
        print(f"  Efficiency: {efficiency:.2f}")
        print()

=== Amdahl's Law Experiment ===

Threads: 1, Time: 1.1145 sec
Threads: 2, Time: 1.2654 sec
Threads: 4, Time: 1.2047 sec
Threads: 8, Time: 1.1764 sec

=== Performance Metrics ===

Threads: 1
  Speedup   : 1.00
  Efficiency: 1.00

Threads: 2
  Speedup   : 0.88
  Efficiency: 0.44

Threads: 4
  Speedup   : 0.93
  Efficiency: 0.23

Threads: 8
  Speedup   : 0.95
  Efficiency: 0.12



Assignment-9

In [ ]:
# Simulating MPI Send/Receive using functions
def send_message(message):
    buffer = message
    return buffer
def receive_message(buffer):
    received = buffer
    return received
def process_0():
    message = "Hello from Process 0"
    buffer = send_message(message)
    return buffer
def process_1(buffer):
    received_message = receive_message(buffer)
    return received_message
def main():
    buffer = process_0()
    received_message = process_1(buffer)
    success = False
    if received_message == "Hello from Process 0":
        success = True
    return received_message, success
msg, status = main()
print("Received Message:", msg)
print("Communication Successful:", status)


Received Message: Hello from Process 0
Communication Successful: True


Assignment-10

In [ ]:
 import multiprocessing as mp
import random
import time

# Worker function (simulates MPI worker process)
def worker_process(worker_id, queue):
    random.seed(time.time() + worker_id)
    temperature = random.randint(20, 40)
    print(f"Worker {worker_id} generated temperature: {temperature}")

    # Send data to master using queue
    queue.put((worker_id, temperature))


if __name__ == "__main__":
    num_workers = 3  # like MPI processes (excluding master)
    queue = mp.Queue()
    processes = []

    print("\nMaster collecting temperatures...\n")

    # Create worker processes
    for i in range(1, num_workers + 1):
        p = mp.Process(target=worker_process, args=(i, queue))
        processes.append(p)
        p.start()

    temperatures = []

    # Receive data from workers
    for _ in range(num_workers):
        worker_id, temp = queue.get()
        print(f"Received from worker {worker_id}: {temp}")
        temperatures.append(temp)

    # Wait for all processes to finish
    for p in processes:
        p.join()

    # Compute results
    avg = sum(temperatures) / len(temperatures)
    max_temp = max(temperatures)

    print("\n--- Results ---")
    print(f"Average Temperature = {avg:.2f}")
    print(f"Maximum Temperature = {max_temp}")


Master collecting temperatures...

Worker 1 generated temperature: 32
Worker 3 generated temperature: 40Worker 2 generated temperature: 32

Received from worker 1: 32
Received from worker 3: 40
Received from worker 2: 32

--- Results ---
Average Temperature = 34.67
Maximum Temperature = 40


Assignment-11

In [ ]:
import multiprocessing as mp
import numpy as np

# Worker function
def worker_process(rank, chunk, scale, queue):
    print(f"Process {rank}: Received chunk {chunk}")

    # Multiply chunk with scaling factor (like computation)
    scaled_chunk = chunk * scale

    print(f"Process {rank}: Scaled chunk {scaled_chunk}")

    # Send result back to master
    queue.put((rank, scaled_chunk))


if __name__ == "__main__":
    num_processes = 4   # simulate 4 MPI processes
    queue = mp.Queue()

    # Rank 0 work (Master)
    print("Rank 0: Generating data...\n")

    data = np.arange(120)   # 120 data points
    scale_factor = 2.5

    print("Original Data:", data)
    print("Scaling Factor:", scale_factor, "\n")

    # 🔹 Scatter (split data into equal chunks)
    chunks = np.array_split(data, num_processes)

    processes = []

    # Start worker processes
    for rank in range(num_processes):
        p = mp.Process(target=worker_process,
                       args=(rank, chunks[rank], scale_factor, queue))
        processes.append(p)
        p.start()

    # 🔹 Gather (collect results)
    results = [None] * num_processes

    for _ in range(num_processes):
        rank, scaled_chunk = queue.get()
        results[rank] = scaled_chunk

    # Wait for all processes
    for p in processes:
        p.join()

    # Reconstruct final array
    final_result = np.concatenate(results)

    print("\n--- Final Scaled Array ---")
    print(final_result)

Rank 0: Generating data...

Original Data: [  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 109 110 111 112 113 114 115 116 117 118 119]
Scaling Factor: 2.5 

Process 0: Received chunk [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29]
Process 1: Received chunk [30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53
 54 55 56 57 58 59]
Process 0: Scaled chunk [ 0.   2.5  5.   7.5 10.  12.5 15.  17.5 20.  22.5 25.  27.5 30.  32.5
 35.  37.5 40.  42.5 45.  47.5 50.  52.5 55.  57.5 60.  62.5 65.  67.5
 70.  72.5]
Process 1: Scaled chunk [ 75.  